# NutriScript AI: Sistema RAG & Dashboard Integral

## Objetivo
Implementar un **Sistema de Generación Aumentada por Recuperación (RAG)** que:
1. Consulte el corpus de recetas saludables
2. Sugiera substituciones de ingredientes más saludables
3. Proporcione insights nutricionales personalizados

## ¿Qué es RAG?
RAG combina **Recuperación** (búsqueda en corpus) + **Generación** (síntesis de respuestas).

En nuestro caso:
- **Recuperación**: Encontrar recetas similares con mejor Nutri-Score
- **Generación**: Sugerir cambios de ingredientes basados en análisis semántico

---

## 1. Setup y Carga

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

# Cargar dataset
df = pd.read_csv('../data/recipes_nlp_processed.csv')

print(f"✓ Dataset cargado: {len(df):,} recetas")
print(f"\nMuestra de datos disponibles:")
print(df[['name', 'ingredients', 'nutriscore', 'calories']].head(3))

---
## 2. Construcción del Índice RAG

### Estrategia de Indexación
Creamos un índice que sea rápido para búsquedas por similitud semántica.

In [ ]:
print("\n🔧 CONSTRUCCIÓN DEL ÍNDICE RAG")
print("="*80)

# Usar TF-IDF en ingredientes para búsquedas semánticas
print(f"\n📊 Creando vectorizador TF-IDF para ingredientes...")

# Limpiar ingredientes (convertir formato string a texto)
import ast
def parse_ingredients(ingredients_str):
    try:
        ingredients_list = ast.literal_eval(ingredients_str)
        return ' '.join([str(ing).lower() for ing in ingredients_list])
    except:
        return ""

df['ingredients_text'] = df['ingredients'].progress_apply(parse_ingredients)

# Vectorizar
vectorizer_rag = TfidfVectorizer(max_features=2000, lowercase=True, stop_words='english')
X_ingredients = vectorizer_rag.fit_transform(df['ingredients_text'])

print(f"✓ Índice creado")
print(f"  - Matriz de ingredientes: {X_ingredients.shape}")
print(f"  - Términos únicos: {X_ingredients.shape[1]}")
print(f"  - Density: {X_ingredients.nnz / (X_ingredients.shape[0] * X_ingredients.shape[1]) * 100:.2f}%")

---
## 3. Función Principal: RAG - Sugerir Substituciones

In [ ]:
def rag_suggest_healthy_ingredient(ingredient, top_n=5, min_nutriscore='B'):
    """
    Sistema RAG: Sugiere substituciones más saludables para un ingrediente.
    
    Parámetros:
    -----------
    ingredient : str
        Ingrediente a substituir (ej: "butter", "sugar", "oil")
    top_n : int
        Número de sugerencias a retornar (default: 5)
    min_nutriscore : str
        Score mínimo deseado (A, B, C, D, E)
    
    Retorna:
    --------
    list : Lista de sugerencias con receta, ingredientes y análisis
    """
    
    # 1. RECUPERACIÓN: Encontrar recetas con el ingrediente
    query_vec = vectorizer_rag.transform([ingredient.lower()])
    similarities = cosine_similarity(query_vec, X_ingredients).flatten()
    
    # Filtrar por Nutri-Score
    score_rank = {'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 1}
    min_rank = score_rank.get(min_nutriscore, 3)
    
    # Combinar similaridad y Nutri-Score
    scores = similarities.copy()
    for i, row in df.iterrows():
        recipe_score_rank = score_rank.get(row['nutriscore'], 1)
        scores[i] *= (recipe_score_rank / 5)  # Peso: buenos scores tienen mayor puntuación
    
    # Top N índices
    top_idx = scores.argsort()[-top_n:][::-1]
    
    # 2. GENERACIÓN: Crear sugerencias con análisis
    suggestions = []
    for idx in top_idx:
        if similarities[idx] > 0.1:  # Threshold de similitud
            row = df.iloc[idx]
            suggestion = {
                'recipe_id': row['id'],
                'recipe_name': row['name'],
                'ingredients_list': row['ingredients'],
                'nutriscore': row['nutriscore'],
                'calories': row['calories'],
                'sugar': row['sugar'],
                'sat_fat': row['sat_fat'],
                'similarity_score': similarities[idx],
                'recommendation': f"Esta receta {row['name']} utiliza ingredientes alternativos con Nutri-Score {row['nutriscore']} (similitud: {similarities[idx]:.2f})"
            }
            suggestions.append(suggestion)
    
    return suggestions

print("✓ Función RAG definida")

In [ ]:
# Ejemplo 1: Butter (grasas saturadas altas)
print("\n" + "="*80)
print("🔍 EJEMPLO 1: Substituir BUTTER (grasas saturadas)")
print("="*80)

suggestions_butter = rag_suggest_healthy_ingredient('butter', top_n=5, min_nutriscore='B')

for i, sugg in enumerate(suggestions_butter, 1):
    print(f"\n{i}. {sugg['recipe_name']}")
    print(f"   Nutri-Score: {sugg['nutriscore']} | Calorías: {sugg['calories']:.0f}")
    print(f"   Grasas Saturadas: {sugg['sat_fat']:.1f}g | Azúcar: {sugg['sugar']:.1f}g")
    print(f"   Similitud con 'butter': {sugg['similarity_score']:.3f}")
    print(f"   💡 {sugg['recommendation']}")

In [ ]:
# Ejemplo 2: Sugar
print("\n" + "="*80)
print("🔍 EJEMPLO 2: Substituir SUGAR (reducir azúcar)")
print("="*80)

suggestions_sugar = rag_suggest_healthy_ingredient('sugar', top_n=5, min_nutriscore='A')

for i, sugg in enumerate(suggestions_sugar, 1):
    print(f"\n{i}. {sugg['recipe_name']}")
    print(f"   Nutri-Score: {sugg['nutriscore']} | Calorías: {sugg['calories']:.0f}")
    print(f"   Azúcar: {sugg['sugar']:.1f}g | Grasas Saturadas: {sugg['sat_fat']:.1f}g")
    print(f"   Similitud con 'sugar': {sugg['similarity_score']:.3f}")
    print(f"   💡 {sugg['recommendation']}")

In [ ]:
# Ejemplo 3: Oil
print("\n" + "="*80)
print("🔍 EJEMPLO 3: Substituir OIL (optimizar grasa)")
print("="*80)

suggestions_oil = rag_suggest_healthy_ingredient('oil', top_n=5, min_nutriscore='B')

for i, sugg in enumerate(suggestions_oil, 1):
    print(f"\n{i}. {sugg['recipe_name']}")
    print(f"   Nutri-Score: {sugg['nutriscore']} | Calorías: {sugg['calories']:.0f}")
    print(f"   Grasas: {sugg['sat_fat']:.1f}g | Azúcar: {sugg['sugar']:.1f}g")
    print(f"   Similitud con 'oil': {sugg['similarity_score']:.3f}")
    print(f"   💡 {sugg['recommendation']}")

---
## 4. Análisis de Ingredientes Más Saludables

In [ ]:
# Analizar ingredientes más comunes en recetas A/B (saludables) vs D/E (menos saludables)

print("\n📊 ANÁLISIS COMPARATIVO DE INGREDIENTES")
print("="*80)

# Extraer ingredientes
healthy_recipes = df[df['nutriscore'].isin(['A', 'B'])]
unhealthy_recipes = df[df['nutriscore'].isin(['D', 'E'])]

def extract_ingredients_list(df_subset):
    all_ingredients = []
    for ing_str in df_subset['ingredients_text']:
        ingredients = ing_str.split()
        all_ingredients.extend(ingredients)
    return Counter(all_ingredients)

healthy_ings = extract_ingredients_list(healthy_recipes)
unhealthy_ings = extract_ingredients_list(unhealthy_recipes)

# Top ingredientes saludables
print(f"\n🟢 Top 15 Ingredientes en Recetas SALUDABLES (A-B):")
top_healthy = healthy_ings.most_common(15)
for i, (ing, count) in enumerate(top_healthy, 1):
    pct = count / sum(healthy_ings.values()) * 100
    print(f"{i:2d}. {ing:20s} ({count:5d} ocurrencias, {pct:5.1f}%)")

# Top ingredientes menos saludables
print(f"\n🔴 Top 15 Ingredientes en Recetas MENOS SALUDABLES (D-E):")
top_unhealthy = unhealthy_ings.most_common(15)
for i, (ing, count) in enumerate(top_unhealthy, 1):
    pct = count / sum(unhealthy_ings.values()) * 100
    print(f"{i:2d}. {ing:20s} ({count:5d} ocurrencias, {pct:5.1f}%)")

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

healthy_names = [ing for ing, _ in top_healthy[:10]]
healthy_counts = [count for _, count in top_healthy[:10]]

unhealthy_names = [ing for ing, _ in top_unhealthy[:10]]
unhealthy_counts = [count for _, count in top_unhealthy[:10]]

axes[0].barh(healthy_names, healthy_counts, color='#4CAF50', edgecolor='black', alpha=0.7)
axes[0].set_title('Top 10 Ingredientes en Recetas Saludables (A-B)', fontweight='bold')
axes[0].set_xlabel('Frecuencia')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(unhealthy_names, unhealthy_counts, color='#F44336', edgecolor='black', alpha=0.7)
axes[1].set_title('Top 10 Ingredientes en Recetas Menos Saludables (D-E)', fontweight='bold')
axes[1].set_xlabel('Frecuencia')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/ingredients_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado")

In [ ]:
# Identificar ingredientes que aparecen MÁS en recetas saludables
print("\n🎯 INGREDIENTES RECOMENDADOS (más en saludables que poco saludables):")
print("="*80)

ingredient_ratio = {}
all_ingredients = set(healthy_ings.keys()) | set(unhealthy_ings.keys())

for ing in all_ingredients:
    healthy_count = healthy_ings.get(ing, 0) + 1  # +1 para evitar división por cero
    unhealthy_count = unhealthy_ings.get(ing, 0) + 1
    ratio = healthy_count / unhealthy_count
    ingredient_ratio[ing] = ratio

# Top 15 ingredientes desproporcionadamente en recetas saludables
sorted_ratio = sorted(ingredient_ratio.items(), key=lambda x: x[1], reverse=True)

print(f"\n✅ TOP 15 - Usar estos ingredientes para mejorar:")
for i, (ing, ratio) in enumerate(sorted_ratio[:15], 1):
    healthy_count = healthy_ings.get(ing, 0)
    unhealthy_count = unhealthy_ings.get(ing, 0)
    print(f"{i:2d}. {ing:20s} (Ratio: {ratio:.1f}x más en saludables)")

---
## 5. Dashboard Analítico Interactivo

In [ ]:
# Dashboard: Análisis de Recetas por Usuario
def create_recipe_health_analysis(recipe_name):
    """
    Dashboard personal: Analiza una receta y sugiere mejoras
    """
    # Encontrar receta
    recipe = df[df['name'].str.lower() == recipe_name.lower()].iloc[0]
    
    print(f"\n" + "="*80)
    print(f"📋 ANÁLISIS DE SALUD: {recipe['name'].upper()}")
    print("="*80)
    
    # Información básica
    print(f"\n📊 INFORMACIÓN NUTRICIONAL:")
    print(f"  - Calorías: {recipe['calories']:.0f}")
    print(f"  - Azúcar: {recipe['sugar']:.1f}g")
    print(f"  - Grasas saturadas: {recipe['sat_fat']:.1f}g")
    print(f"  - Proteína: {recipe['protein']:.1f}g")
    print(f"  - Sodio: {recipe['sodium']:.1f}mg")
    
    print(f"\n🏆 PUNTUACIÓN:")
    score_colors = {'A': '🟢', 'B': '🟡', 'C': '🟠', 'D': '🔴', 'E': '⚫'}
    score_desc = {'A': 'Excelente', 'B': 'Bueno', 'C': 'Moderado', 'D': 'Pobre', 'E': 'Muy Pobre'}
    print(f"  Nutri-Score: {score_colors[recipe['nutriscore']]} {recipe['nutriscore']} - {score_desc[recipe['nutriscore']]}")
    print(f"  Dificultad: {recipe['difficulty']}")
    print(f"  Tipo de Dieta: {recipe['diet_type']}")
    
    # Sugerencias de mejora
    print(f"\n💡 RECOMENDACIONES DE MEJORA:")
    
    issues = []
    if recipe['sugar'] > 10:
        issues.append(f"  ⚠️ Azúcar elevada ({recipe['sugar']:.1f}g). Considera reducir endulzantes.")
    if recipe['sat_fat'] > 6:
        issues.append(f"  ⚠️ Grasas saturadas altas ({recipe['sat_fat']:.1f}g). Usa aceite de oliva instead.")
    if recipe['calories'] > 800:
        issues.append(f"  ⚠️ Receta muy calórica ({recipe['calories']:.0f}). Considera porciones más pequeñas.")
    if recipe['sodium'] > 1500:
        issues.append(f"  ⚠️ Sodio elevado ({recipe['sodium']:.0f}mg). Reduce sal.")
    
    if issues:
        for issue in issues:
            print(issue)
    else:
        print("  ✓ ¡Receta muy saludable! Sin recomendaciones críticas.")
    
    # Sugerencias RAG
    print(f"\n🔍 RECETAS ALTERNATIVAS MÁS SALUDABLES:")
    
    # Usar ingredientes clave
    ingredients_text = recipe['ingredients_text'].split()
    if ingredients_text:
        main_ing = ingredients_text[0]  # Primer ingrediente como referencia
        suggestions = rag_suggest_healthy_ingredient(main_ing, top_n=3, min_nutriscore='B')
        
        for i, sugg in enumerate(suggestions[:3], 1):
            print(f"\n  {i}. {sugg['recipe_name']} (Nutri-Score: {sugg['nutriscore']})")
            print(f"     Calorías: {sugg['calories']:.0f} | Azúcar: {sugg['sugar']:.1f}g | Grasas: {sugg['sat_fat']:.1f}g")

print("\n✓ Función de análisis creada")

In [ ]:
# Ejemplo de análisis
sample_recipe = df['name'].iloc[100]  # Receta aleatoria
create_recipe_health_analysis(sample_recipe)

---
## 6. Estadísticas de Impacto RAG

In [ ]:
# Calcular potencial de mejora si seguimos recommendations
print("\n" + "="*80)
print("📈 IMPACTO POTENCIAL DEL SISTEMA RAG")
print("="*80)

# Recetas D/E (menos saludables)
unhealthy = df[df['nutriscore'].isin(['D', 'E'])]
healthy = df[df['nutriscore'].isin(['A', 'B'])]

print(f"\n📊 ESTADO ACTUAL:")
print(f"  - Recetas saludables (A-B): {len(healthy):,} ({len(healthy)/len(df)*100:.1f}%)")
print(f"  - Recetas menos saludables (D-E): {len(unhealthy):,} ({len(unhealthy)/len(df)*100:.1f}%)")

print(f"\n🎯 OPORTUNIDAD DE MEJORA:")
print(f"  - Promedio azúcar en D/E: {unhealthy['sugar'].mean():.1f}g")
print(f"  - Promedio azúcar en A/B: {healthy['sugar'].mean():.1f}g")
print(f"  - Reducción potencial: -{unhealthy['sugar'].mean() - healthy['sugar'].mean():.1f}g por receta")

print(f"\n  - Promedio grasas saturadas en D/E: {unhealthy['sat_fat'].mean():.1f}g")
print(f"  - Promedio grasas saturadas en A/B: {healthy['sat_fat'].mean():.1f}g")
print(f"  - Reducción potencial: -{unhealthy['sat_fat'].mean() - healthy['sat_fat'].mean():.1f}g por receta")

print(f"\n  - Promedio calorías en D/E: {unhealthy['calories'].mean():.0f}")
print(f"  - Promedio calorías en A/B: {healthy['calories'].mean():.0f}")
print(f"  - Reducción potencial: -{unhealthy['calories'].mean() - healthy['calories'].mean():.0f} calorías por receta")

print(f"\n✅ El sistema RAG puede ayudar a {len(unhealthy):,} usuarios a mejorar su nutrición")

In [ ]:
# Visualización de impacto
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

categories = ['D-E\n(Actual)', 'A-B\n(Objetivo)']
colors = ['#F44336', '#4CAF50']

# Azúcar
sugar_vals = [unhealthy['sugar'].mean(), healthy['sugar'].mean()]
axes[0, 0].bar(categories, sugar_vals, color=colors, edgecolor='black', alpha=0.7, width=0.6)
axes[0, 0].set_title('Azúcar (g)', fontweight='bold')
axes[0, 0].set_ylabel('Gramos')
for i, v in enumerate(sugar_vals):
    axes[0, 0].text(i, v + 0.5, f"{v:.1f}g", ha='center', fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Grasas saturadas
sat_fat_vals = [unhealthy['sat_fat'].mean(), healthy['sat_fat'].mean()]
axes[0, 1].bar(categories, sat_fat_vals, color=colors, edgecolor='black', alpha=0.7, width=0.6)
axes[0, 1].set_title('Grasas Saturadas (g)', fontweight='bold')
axes[0, 1].set_ylabel('Gramos')
for i, v in enumerate(sat_fat_vals):
    axes[0, 1].text(i, v + 0.2, f"{v:.1f}g", ha='center', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Calorías
cal_vals = [unhealthy['calories'].mean(), healthy['calories'].mean()]
axes[1, 0].bar(categories, cal_vals, color=colors, edgecolor='black', alpha=0.7, width=0.6)
axes[1, 0].set_title('Calorías', fontweight='bold')
axes[1, 0].set_ylabel('kcal')
for i, v in enumerate(cal_vals):
    axes[1, 0].text(i, v + 20, f"{v:.0f}", ha='center', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# Distribución Nutri-Score
nutriscore_dist = df['nutriscore'].value_counts().sort_index()
score_colors_chart = ['#4CAF50', '#8BC34A', '#FFC107', '#FF9800', '#F44336']
axes[1, 1].pie(nutriscore_dist.values, labels=nutriscore_dist.index, colors=score_colors_chart,
              autopct='%1.1f%%', startangle=90)
axes[1, 1].set_title('Distribución de Nutri-Scores', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/rag_impact_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 7. Integración Final: Exportar Datos para Dashboard Streamlit

In [ ]:
# Guardar índice RAG para uso en Streamlit
import pickle
import joblib

# Guardar metadatos para consultas futuras
rag_metadata = {
    'vectorizer': vectorizer_rag,
    'df': df[['name', 'id', 'ingredients_text', 'nutriscore', 'calories', 'sugar', 'sat_fat', 'protein', 'sodium']],
    'X_ingredients': X_ingredients
}

joblib.dump(rag_metadata, '../data/rag_index.pkl')
print("✓ Índice RAG guardado: ../data/rag_index.pkl")

# Dataset final para dashboard
df_dashboard = df[[
    'id', 'name', 'minutes', 'n_steps', 'n_ingredients',
    'calories', 'sugar', 'sat_fat', 'protein', 'sodium',
    'nutriscore', 'difficulty', 'diet_type'
]]
df_dashboard.to_csv('../data/recipes_dashboard.csv', index=False)
print("✓ Dataset para dashboard guardado: ../data/recipes_dashboard.csv")

---
## 8. Resumen Final del Proyecto

In [ ]:
print("\n" + "="*80)
print("🎉 RESUMEN EJECUTIVO - PROYECTO NUTRISRIPT AI")
print("="*80)

print(f"\n1️⃣ DATOS Y PREPROCESAMIENTO:")
print(f"   ✓ {len(df):,} recetas procesadas")
print(f"   ✓ Features extraídos: lemas, verbos, sintagmas nominales")
print(f"   ✓ Targets creados: Nutri-Score (A-E), Dificultad, Tipo de Dieta")

print(f"\n2️⃣ REPRESENTACIÓN SEMÁNTICA:")
print(f"   ✓ TF-IDF: {X_tfidf.shape[1]:,} features (5000 términos)")
print(f"   ✓ SVD/LSA: Reducción a 100 componentes")
print(f"   ✓ TPF-IDF para RAG: {X_ingredients.shape[1]:,} features (2000 términos)")

print(f"\n3️⃣ MODELOS DE MACHINE LEARNING:")
print(f"   ✓ Baseline (Naive Bayes): Accuracy {test_acc:.3f}")
print(f"   ✓ Deep Learning (LSTM): Accuracy {test_acc_lstm:.3f}")
print(f"   ✓ Mejora: +{(test_acc_lstm - test_acc)*100:.1f}%")

print(f"\n4️⃣ SISTEMA RAG:")
print(f"   ✓ Índice semántico de {len(df):,} recetas")
print(f"   ✓ Búsqueda por similitud de ingredientes")
print(f"   ✓ Recomendaciones personalizadas de substitución")
print(f"   ✓ Usuarios potenciales: {len(unhealthy):,} recetas menos saludables")

print(f"\n5️⃣ OUTPUTS GENERADOS:")
print(f"   ✓ notebooks/01_Text_Mining_EDA.ipynb")
print(f"   ✓ notebooks/02_NLP_Pipeline_Preprocessing.ipynb")
print(f"   ✓ notebooks/03_Model_Training_TfIdf_NB_LSTM.ipynb")
print(f"   ✓ notebooks/04_RAG_System_Dashboard.ipynb")

print(f"\n6️⃣ MODELOS GUARDADOS:")
print(f"   ✓ lstm_nutriscore_model.h5")
print(f"   ✓ naive_bayes_model.pkl")
print(f"   ✓ rag_index.pkl (para búsquedas)")
print(f"   ✓ Datasets procesados (.csv) para dashboard")

print(f"\n7️⃣ APLICACIÓN:")
print(f"   ✓ FastAPI: app/main.py (endpoints REST)")
print(f"   ✓ Streamlit: app/dashboard.py (frontend interactivo)")
print(f"   ✓ Docker: docker/Dockerfile (despliegue containerizado)")

print(f"\n8️⃣ TECNOLOGÍAS UTILIZADAS:")
print(f"   ✓ NLP: spaCy (lematización, POS tagging)")
print(f"   ✓ ML: scikit-learn (TF-IDF, SVD, Naive Bayes)")
print(f"   ✓ DL: TensorFlow/Keras (LSTM)")
print(f"   ✓ Web: FastAPI, Streamlit")
print(f"   ✓ Data: pandas, numpy")

print(f"\n" + "="*80)
print("✅ PROYECTO COMPLETADO CON ÉXITO")
print("="*80)